In [9]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)

print("All required libraries imported successfully.")

All required libraries imported successfully.


In [10]:
cancer = load_breast_cancer(as_frame=True)
df = cancer.frame

num_instances = df.shape[0]
num_features = df.shape[1] - 1

print("=== DATASET SPECIFICATION VERIFICATION ===")
print(f"Total Instances (Rows) : {num_instances} (Criteria: >= 500) -> {'PASS' if num_instances >= 500 else 'FAIL'}")
print(f"Total Features (Cols)  : {num_features} (Criteria: >= 12)  -> {'PASS' if num_features >= 12 else 'FAIL'}")
print("\nTarget Class Distribution:")
print(df['target'].value_counts())

# Save full dataset copy
df.to_csv("dataset.csv", index=False)
print("\n'dataset.csv' created successfully.")

=== DATASET SPECIFICATION VERIFICATION ===
Total Instances (Rows) : 569 (Criteria: >= 500) -> PASS
Total Features (Cols)  : 30 (Criteria: >= 12)  -> PASS

Target Class Distribution:
target
1    357
0    212
Name: count, dtype: int64

'dataset.csv' created successfully.


In [11]:
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Export test data with features and target for the Streamlit file uploader
test_df = X_test.copy()
test_df['target'] = y_test
test_df.to_csv("test_data.csv", index=False)

# Preprocessing: Fit scaler on training set, transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Directory for serialized models
os.makedirs("model", exist_ok=True)
joblib.dump(scaler, "model/scaler.pkl")

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape : {X_test.shape}")
print("[SUCCESS] 'test_data.csv' and 'model/scaler.pkl' generated.")

Training set shape: (455, 30)
Testing set shape : (114, 30)
[SUCCESS] 'test_data.csv' and 'model/scaler.pkl' generated.


In [12]:
from sklearn.calibration import CalibratedClassifierCV

# Define models using CalibratedClassifierCV for clean probability calibration
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": CalibratedClassifierCV(
        SVC(kernel='rbf', random_state=42),
        ensemble=False
    )
}

results = []

for name, clf in models.items():
    if name in ["Logistic Regression", "KNN", "Support Vector Machine"]:
        clf.fit(X_train_scaled, y_train)
        preds = clf.predict(X_test_scaled)
        probs = clf.predict_proba(X_test_scaled)[:, 1]
    else:
        clf.fit(X_train, y_train)
        preds = clf.predict(X_test)
        probs = clf.predict_proba(X_test)[:, 1]

    # Calculate 6 evaluation metrics
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    mcc = matthews_corrcoef(y_test, preds)

    results.append({
        "ML Model Name": name,
        "Accuracy": round(acc, 4),
        "AUC": round(auc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4),
        "MCC": round(mcc, 4)
    })

    # Save trained model artifact
    filename = name.lower().replace(" ", "_") + ".pkl"
    joblib.dump(clf, f"model/{filename}")

# Print final comparison table
metrics_df = pd.DataFrame(results)
print("=== FINAL EVALUATION METRICS TABLE ===\n")
print(metrics_df.to_string(index=False))

print("\nAll 5 models trained and saved under 'model/' directory.")

=== FINAL EVALUATION METRICS TABLE ===

         ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
   Logistic Regression    0.9825 0.9954     0.9861  0.9861 0.9861 0.9623
         Decision Tree    0.9211 0.9163     0.9565  0.9167 0.9362 0.8341
                   KNN    0.9561 0.9788     0.9589  0.9722 0.9655 0.9054
           Naive Bayes    0.9386 0.9878     0.9452  0.9583 0.9517 0.8676
Support Vector Machine    0.9825 0.9950     0.9861  0.9861 0.9861 0.9623

All 5 models trained and saved under 'model/' directory.
